In [2]:
import time
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

In [3]:
class CrashState(TypedDict):
    step1: str
    step2: str
    step3: str

In [4]:
def step1(state: CrashState):
    print('Step 1 reached and completed')
    return {'step1': 'success'}


def step2(state: CrashState):
    print('Step 2 reached and program paused. Keyboard interrup to simulate crash')
    time.sleep(10)
    return {'step2': 'success'}


def step3(state: CrashState):
    print('Step 3 reached and completed')
    return {'step3': 'success'}

In [5]:
graph = StateGraph(CrashState)

graph.add_node('step1', step1)
graph.add_node('step2', step2)
graph.add_node('step3', step3)

graph.add_edge(START, 'step1')
graph.add_edge('step1', 'step2')
graph.add_edge('step2', 'step3')
graph.add_edge('step3', END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config = {'configurable': {'thread_id': 1}}
workflow.invoke({}, config=config)

Step 1 reached and completed
Step 2 reached and program paused. Keyboard interrup to simulate crash


KeyboardInterrupt: 

Stopped at step2. So when invoking graph with same thread_id again, it should continue from step 2. This is because of the "next" key in the saved state checkpoint. To resume though, you have to give None as invoke input.

In [10]:
workflow.invoke(input=None, config=config)

Step 2 reached and program paused. Keyboard interrup to simulate crash
Step 3 reached and completed


{'step1': 'success', 'step2': 'success', 'step3': 'success'}